# Jak komputer uczy się rozpoznawać rysunki?

Uruchamiaj komórki po kolei: zaznacz komórkę i naciśnij **Shift+Enter**.

In [ ]:
import os, urllib.request
REPO = "https://rodakt.github.io/obrazy-cnn"  # strona z materiałami
if not os.path.exists("warsztat.py"):  # kod pomocniczy pobiera się raz, obok notatnika
    urllib.request.urlretrieve(REPO + "/warsztat.py", "warsztat.py.tmp")
    os.replace("warsztat.py.tmp", "warsztat.py")  # przerwane pobieranie nie zostawi uszkodzonego pliku
import warsztat as w
w.REPO = REPO  # stąd też zapasowa kopia danych

## 1. Obraz to liczby

Komputer nie widzi obrazków tak jak my. Dla niego obraz to tabela liczb.

▶ Uruchom komórkę, narysuj w okienku cokolwiek i kliknij **Gotowe**.
*Nie widać okna? Może być schowane za przeglądarką — sprawdź pasek zadań.*

In [ ]:
rysunek = w.narysuj()
w.pokaz_liczby(rysunek)

**Do zastanowienia**

- Jaką liczbą zapisane jest puste tło, a jaką pełna kreska?
- Twój rysunek został przycięty i zmniejszony do 28 × 28 pikseli. Ile liczb go opisuje?
  A ile liczb opisuje zdjęcie z telefonu (12 mln pikseli, każdy ma 3 kolory)?

## 2. Dane: rysunki z gry Quick, Draw!

W grze [Quick, Draw!](https://quickdraw.withgoogle.com) gracz ma 20 sekund na narysowanie hasła, a komputer zgaduje, co to jest.
Ludzie z całego świata narysowali w niej około 50 milionów rysunków. Google udostępnił je za darmo.

Bierzemy po 2000 rysunków z pięciu kategorii, każdy ma 28×28 pikseli.

🔒 Z każdej kategorii **500 rysunków chowamy**: sieć nie zobaczy ich podczas nauki.
Na nich sprawdzimy, czy rozpoznaje nowe rysunki, a nie tylko pamięta stare.

In [ ]:
dane = w.pobierz(["apple", "house", "bicycle", "fish", "star"])
w.pokaz_przyklady(dane)

Każdy z tych rysunków to dla komputera też 784 liczby. Oto pierwszy z nich:

In [ ]:
w.pokaz_liczby(dane.Xtr[0])

## 3. Splot: szukanie wzorców

Sieć konwolucyjna nie ogląda całego rysunku naraz. Przesuwa po nim małe okienko — **filtr**.
W każdym miejscu sprawdza, jak bardzo fragment rysunku pasuje do wzorca zapisanego w filtrze.
Ta operacja to **splot** (*konwolucja*) — stąd nazwa sieci.

Filtr to też tabelka liczb. Ten poniżej szuka pionowych kresek. Wypróbuj go na rysunku z części 1:

In [ ]:
w.pokaz_splot(rysunek, "pionowe")

▶ Zmień `"pionowe"` na `"poziome"`, `"ukośne"`, `"ukośne2"` albo `"rozmycie"` i uruchom komórkę jeszcze raz.

▶ Rachunek dla innego miejsca rysunku: dopisz `miejsce=(wiersz, kolumna)`,
np. `w.pokaz_splot(rysunek, "pionowe", miejsce=(10, 14))`.

Filtr można też wpisać samemu:

In [ ]:
w.pokaz_splot(rysunek, [[1, 0, 1],
                        [0, 0, 0],
                        [1, 0, 1]])

**Do zastanowienia**

- Co robi filtr z jedną jedynką pośrodku? A co się stanie, gdy jedynkę przesuniesz w róg tabelki?
- Liczby w filtrze `"pionowe"` sumują się do zera. Co dzięki temu wychodzi tam, gdzie rysunek jest pusty — i tam, gdzie byłby cały zamalowany?

## 4. Sieć konwolucyjna (LeNet-5)

W prawdziwej sieci nikt nie wpisuje filtrów ręcznie. Nasza ma dwie warstwy splotowe: 6 filtrów 5×5 i 16 kolejnych.
Liczby w filtrach (*wagi*) są na początku **losowe**. Sieć sama dobierze je podczas uczenia.

In [ ]:
model = w.LeNet(len(dane.klasy))
w.pokaz_siec(model, rysunek, dane)

Twój rysunek z części 1 przechodzi przez sieć od lewej do prawej. Sieć jeszcze się nie uczyła,
więc jej odpowiedź jest przypadkowa.

Tak wyglądają losowe filtry pierwszej warstwy i to, co znajdują na Twoim rysunku:

In [ ]:
w.pokaz_filtry(model, rysunek)

Co nienauczona sieć mówi o rysunkach testowych?

In [ ]:
w.pokaz_test(model, dane)

**Do zastanowienia**

- Sieć prawie na każdy rysunek odpowiada to samo. Dlaczego trafność wychodzi wtedy około 20%?

## 5. Uczenie

Sieć ogląda rysunki treningowe porcjami po 128. Po każdej porcji:

1. sprawdza, jak bardzo się pomyliła — tę miarę błędu nazywamy **stratą**;
2. trochę poprawia wagi (także liczby w filtrach), żeby strata zmalała.

Jedno przejście przez wszystkie rysunki treningowe to **epoka**.

Wykres rysuje się na żywo. Po lewej strata, po prawej trafność na schowanych rysunkach testowych.

In [ ]:
w.trenuj(model, dane, epoki=5)

**Do zastanowienia**

- W którym momencie sieć uczy się najszybciej? Ile epok faktycznie było potrzebnych?
- Uruchom komórkę z uczeniem jeszcze raz — sieć uczy się dalej od miejsca, w którym skończyła. Co się dzieje ze stratą i trafnością?

## 6. Sprawdzenie na rysunkach, których sieć nie widziała

🔓 Czas zdjąć kłódkę. Oto rysunki schowane w części 2 i odpowiedzi sieci:

In [ ]:
w.odkryj(model, dane)

Więcej losowych rysunków testowych: zielone — sieć zgadła, czerwone — pomyliła się.

▶ Uruchom komórkę kilka razy, żeby zobaczyć inne.

In [ ]:
w.pokaz_test(model, dane)

## 7. Gdzie sieć się myli?

Sama trafność nie mówi, *jakie* błędy robi sieć. Pokaże to *macierz pomyłek*:

- wiersz — co naprawdę było na rysunku,
- kolumna — co odpowiedziała sieć.

Najedź myszą na komórkę, żeby zobaczyć opis.

In [ ]:
w.macierz_pomylek(model, dane)

A to rysunki, przy których sieć pomyliła się, będąc najbardziej pewną swego:

In [ ]:
w.pokaz_pomylki(model, dane)

▶ Pomyłki jednego rodzaju: `w.pokaz_pomylki(model, dane, bylo="ryba", odpowiedz="rower")`.
Wystarczy też samo `bylo=...` albo samo `odpowiedz=...`.

**Do zastanowienia**

- Które dwie kategorie sieć myli najczęściej? Co mają wspólnego ich rysunki?
- Obejrzyj najbardziej „pewne” pomyłki. Przy których rysunkach człowiek też miałby kłopot?

## 8. Twoje rysunki

Co sieć powie o rysunku z części 1?

In [ ]:
w.rozpoznaj(model, rysunek, dane)

Teraz narysuj jedną z pięciu kategorii i kliknij **Gotowe**:

In [ ]:
nowy = w.narysuj()
w.rozpoznaj(model, nowy, dane)

Kilka rysunków naraz — po każdym kliknij **Gotowe**:

In [ ]:
rysunki = w.narysuj(ile=3)
w.rozpoznaj(model, rysunki, dane)

## 9. Oszukaj sieć

Sieć widziała tylko rysunki pięciu kategorii. Sprawdzimy dwa jej słabe miejsca.

### Bazgroły

Komputer narysował losowe bazgroły. Żaden z nich nie jest jabłkiem, domem, rowerem, rybą ani gwiazdą. Co na to sieć?

In [ ]:
w.bazgroly(model, dane)

▶ Uruchom komórkę jeszcze raz — bazgroły będą inne.

▶ Narysuj własny bazgrół: cokolwiek, co nie jest żadną z pięciu kategorii.

In [ ]:
bazgrol = w.narysuj()
w.rozpoznaj(model, bazgrol, dane)

**Do zastanowienia**

- Czy sieć kiedykolwiek odpowiada „nie wiem”? Czy w ogóle może? (Ile razem dają wszystkie paski prawdopodobieństw?)

### Obrócona ryba

▶ Narysuj rybę **poziomo, jednym pociągnięciem** (bez odrywania przycisku myszy) i kliknij **Gotowe**.

Komórka obróci rysunek co 15°, aż do góry nogami, i pokaże odpowiedź sieci dla każdego kąta.

In [ ]:
ryba = w.narysuj()
w.obracaj(model, ryba, dane)

Zielony podpis — sieć widzi rybę, czerwony — coś innego.
Linia przerywana to średnia z 300 obróconych ryb testowych: to nie kaprys jednego rysunku.

**Do zastanowienia**

- Dla człowieka obrócona ryba to nadal ryba. Dlaczego nie dla sieci? Jak wyglądały ryby w danych z części 2?
- Przy 180° sieć znów widzi rybę. Dlaczego? (Obróć w myślach rybę do góry nogami.)

## 10. Co sieć widzi?

W części 4 filtry były losowe. Podczas uczenia sieć sama je poprawiała — nikt jej nie mówił, czego szukać.
Poniżej mapy obu warstw splotowych dla Twojego rysunku: górny wiersz przed uczeniem, dolny po uczeniu.

In [ ]:
w.pokaz_mapy(model, rysunek)

▶ To samo dla innego rysunku. Zamiast `nowy` wstaw np. `rysunki[0]` albo `dane.Xte[0]`:

In [ ]:
w.pokaz_mapy(model, nowy)

**Do zastanowienia**

- Mapy drugiej warstwy mają tylko 10×10 pikseli. Czy jeszcze rozpoznajesz na nich swój rysunek?

## 11. Od małej sieci do dużych modeli

**Co dziś zobaczyliśmy**

- Obraz to dla komputera tabela liczb (część 1).
- Splot przesuwa po obrazie filtr i szuka w nim wzorca (część 3).
- Sieć sama dobiera wagi: zaczyna od losowych i poprawia je tak, żeby strata malała (części 4–5).
- Uczciwy sprawdzian to rysunki, których sieć nie widziała (część 6).
- Sieć umie tylko to, czego nauczyły ją dane: nie powie „nie wiem”, nie rozpozna obróconej ryby (część 9).

### Ten sam pomysł, większa skala

Nasza sieć to **LeNet-5** z 1998 roku (Yann LeCun z zespołem). Jej odmiany czytały w bankach cyfry na czekach.

W 2012 roku sieć **AlexNet** wygrała konkurs ImageNet — rozpoznawanie zdjęć z 1000 kategorii — z ogromną przewagą.
Miała te same klocki co nasza, tylko w innej skali:

| | nasza sieć | AlexNet (2012) |
|---|---|---|
| obrazy do nauki | 7 500 rysunków 28×28 | 1,2 miliona kolorowych zdjęć 224×224 |
| kategorie | 5 | 1000 |
| czas uczenia | poniżej minuty na zwykłym komputerze | 5–6 dni na dwóch kartach graficznych |

Te sieci i trzecia, ResNet, w jednej skali — każdy prostokąt to jedna warstwa, kolory jak w części 4:

In [ ]:
w.pokaz_skale()

**ResNet** (2015) ma 152 warstwy, a wag mniej więcej tyle co AlexNet.
Tak głęboką sieć dało się nauczyć dzięki *skrótom* (łuki na rysunku): sygnał przeskakuje kilka warstw.

Co się zmieniło od czasów LeNetu? Więcej danych, karty graficzne (tysiące mnożeń naraz) i więcej warstw.
Największe dzisiejsze modele — także te, które piszą teksty i tworzą obrazy — mają miliardy wag.
Uczą się jednak tak samo jak nasza: od losowych wag, krok po kroku, żeby zmniejszyć stratę.

### Słabości też zostają

- **Obrócona ryba.** Pomaga *augmentacja*: sieć ogląda te same obrazy obrócone, przycięte, odbite.
  Nasza sieć uczona na obróconych rysunkach rozpoznaje rybę obróconą o 90° w ok. 80–90% przypadków zamiast 1%.
- **Bazgroły.** Duże sieci też bywają pewne swego na obrazkach bez sensu — nawet na ponad 99%.
  Jak nauczyć sieć mówić „nie wiem”, wciąż nie wiadomo.
- **Dane.** Sieć uczy się także skrzywień w danych. Nasza uznała, że ryba płynie poziomo.
  Przy zdjęciach medycznych takie ukryte skrzywienie może być groźne.

**Do zastanowienia**

- Gdzie na co dzień spotykasz sieci, które rozpoznają obrazy?
- Jaki błąd podobny do obróconej ryby mogłaby popełnić sieć w samochodzie, który sam jeździ?

### Co dalej

- **Inne kategorie.** W części 2 zamień nazwy kategorii i uruchom notatnik od początku.
  Jest ich 345 ([lista](https://github.com/googlecreativelab/quickdraw-dataset/blob/master/categories.txt)),
  po angielsku, np. `"cat"`, `"dog"`, `"hot dog"`. Kot i pies to dla sieci trudna para.
- **Zagraj w [Quick, Draw!](https://quickdraw.withgoogle.com)** i sprawdź, czy sieć Google zgadnie Twoje rysunki.
  Ona nie patrzy na gotowy obrazek, tylko na kolejne kreski w kolejności rysowania.
- **W domu.** Notatnik działa wszędzie, gdzie jest Python z pakietem numpy i Jupyter.
  Instrukcja: [rodakt.github.io/obrazy-cnn](https://rodakt.github.io/obrazy-cnn).

---

Rysunki: *Quick, Draw! Dataset*, Google Creative Lab, licencja [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/),
[github.com/googlecreativelab/quickdraw-dataset](https://github.com/googlecreativelab/quickdraw-dataset).